- **Name:** 20.1_structured_streaming
- **Author:** Shamas Imran
- **Desciption:** Basics of Structured Streaming in Spark
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Created streaming DataFrame from files  
                                              Defined query with writeStream  
                                              Used checkpointing for fault tolerance  
-->

In [1]:
# Root path of your Unity Catalog volume
rootPath = "abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/"

# Master folder for streaming project
masterPath = rootPath + "spark-streaming/"

# Delete recursively
# dbutils.fs.rm(masterPath, recurse=True)

# Define subfolders inside master
inputPath = masterPath + "csv_input"
checkpointPath = masterPath + "checkpoints/csv_query"
outputPath = masterPath + "csv_output"

StatementMeta(, 98d12967-de31-48fd-a609-385ca049dcd9, 3, Finished, Available, Finished)

In [2]:
# Check if folder exists before deleting
if mssparkutils.fs.exists(masterPath):
    mssparkutils.fs.rm(masterPath, True)  # True = recursive delete

StatementMeta(, 98d12967-de31-48fd-a609-385ca049dcd9, 4, Finished, Available, Finished)

In [3]:
import os

# Create directories
os.makedirs(masterPath, exist_ok=True)
os.makedirs(inputPath, exist_ok=True)
os.makedirs(checkpointPath, exist_ok=True)
os.makedirs(outputPath, exist_ok=True)

print("Master folder:", masterPath)
print("Input folder:", inputPath)
print("Checkpoint folder:", checkpointPath)
print("Output folder:", outputPath)


StatementMeta(, 98d12967-de31-48fd-a609-385ca049dcd9, 5, Finished, Available, Finished)

Master folder: abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/spark-streaming/
Input folder: abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/spark-streaming/csv_input
Checkpoint folder: abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/spark-streaming/checkpoints/csv_query
Output folder: abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/spark-streaming/csv_output


In [4]:
import os
import pandas as pd

# Define Lakehouse folder path (adjust if needed)
lakehouse_folder = inputPath

# Create folder if it doesn’t exist
os.makedirs(lakehouse_folder, exist_ok=True)

# Define data for batch01
batch01_data = {
    "id": [1, 2, 3],
    "name": ["Ali", "Sara", "Imran"],
    "score": [85, 90, 70],
    "event_time": [
        "2025-08-18 12:00:05",
        "2025-08-18 12:00:08",
        "2025-08-18 12:00:10"
    ]
}
batch01_df = pd.DataFrame(batch01_data)
batch01_path = f"{lakehouse_folder}/batch01_students.csv"
batch01_df.to_csv(batch01_path, index=False)
print(f"Created: {batch01_path}")

# Define data for batch02
batch02_data = {
    "id": [4, 5],
    "name": ["Ayesha", "Hassan"],
    "score": [95, 88],
    "event_time": [
        "2025-08-18 12:01:02",
        "2025-08-18 12:01:30"
    ]
}
batch02_df = pd.DataFrame(batch02_data)
batch02_path = f"{lakehouse_folder}/batch02_students.csv"
batch02_df.to_csv(batch02_path, index=False)
print(f"Created: {batch02_path}")


StatementMeta(, 98d12967-de31-48fd-a609-385ca049dcd9, 6, Finished, Available, Finished)

Created: abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/spark-streaming/csv_input/batch01_students.csv
Created: abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/spark-streaming/csv_input/batch02_students.csv


In [6]:
for stream in spark.streams.active:
    print(stream.id, stream.name, stream.status)

for q in spark.streams.active:
    q.stop()

for stream in spark.streams.active:
    print(stream.id, stream.name, stream.status)


StatementMeta(, 98d12967-de31-48fd-a609-385ca049dcd9, 8, Finished, Available, Finished)

In [7]:
# 2) Define schema for the incoming CSVs
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("event_time", TimestampType(), True)  # keep for future (watermark/windows)
])

StatementMeta(, 98d12967-de31-48fd-a609-385ca049dcd9, 9, Finished, Available, Finished)

In [9]:
# 3) Create the streaming DataFrame (CSV source)
df_stream = (
    spark.readStream
         .option("header", "true")   # CSV has a header row
         .schema(schema)             # schema is required for streaming
         .csv(inputPath)             # watches this folder for new files
)

StatementMeta(, 98d12967-de31-48fd-a609-385ca049dcd9, 11, Finished, Available, Finished)

In [10]:
# 4) Start a basic console sink (no windows, no watermark)
query = (
    df_stream.writeStream
             .format("csv")
             .option("path", outputPath)
             .option("checkpointLocation", checkpointPath)  # enables recovery/fault tolerance
             .outputMode("append")                          # append since no aggregations
             .trigger(once=True) # .trigger(processingTime="1 seconds")
             .start()
)

query.awaitTermination()

# trigger() ===>
# Default → real-time continuous streaming.
# Processing time → when you want controlled batch frequency.
# Once → for testing or one-off processing.
# AvailableNow → catch-up to current state, then stop.

# outputMode("append") ======>
# Use append if: new rows keep arriving, and you don’t need to rewrite or update old results.
# Use update if: you’re aggregating and only want changes since the last batch.
# Use complete if: you’re aggregating and need the full table every batch.

StatementMeta(, 98d12967-de31-48fd-a609-385ca049dcd9, 12, Finished, Available, Finished)

In [8]:
# 5) (Optional) Stop the stream when you’re done
# for q in spark.streams.active:
#     q.stop()

for stream in spark.streams.active:
    print(stream.id, stream.name, stream.status)


StatementMeta(, bb82ff14-0b76-4e77-848a-9e8d724ea9b4, 10, Finished, Available, Finished)

In [0]:
'''
batch01_students.csv
id,name,score,event_time
1,Ali,85,2025-08-18 12:00:05
2,Sara,90,2025-08-18 12:00:08
3,Imran,70,2025-08-18 12:00:10

batch02_students.csv
id,name,score,event_time
4,Ayesha,95,2025-08-18 12:01:02
5,Hassan,88,2025-08-18 12:01:30
'''